# 01 – Laste og få oversikt over NHIS 2024 (Sample Adult)

Mål: laste dataene, sjekke størrelse og typer, og finne søvnvariablene. Ingen rensing ennå.

In [ ]:
import sys
from pathlib import Path

# Make `src` importable when the notebook is run from notebooks/
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from src.data import load_nhis
from src.config import NHIS_FILE

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

## 1. Last inn
`load_nhis()` laster ned filen fra CDC første gang (ca. 3,6 MB zip) og lagrer den i `data/raw/`. Neste gang brukes den lokale filen.

In [ ]:
df = load_nhis()
print(NHIS_FILE)
print(df.shape)   # forventer ca. 32 629 rader og ca. 630 kolonner

## 2. Første titt

In [ ]:
df.head()

In [ ]:
df.info(max_cols=0)

In [ ]:
# Typer: hvor mange kolonner av hver type?
df.dtypes.value_counts()

## 3. Finn søvnvariablene
Vi gjetter ikke navnene: vi søker i kolonnenavnene og sjekker mot NHIS-kodeboken (variable summary / codebook fra CDC).

In [ ]:
sleep_cols = [c for c in df.columns if "SLP" in c.upper() or "SLEEP" in c.upper()]
sleep_cols

In [ ]:
for c in sleep_cols:
    print(f"--- {c}")
    print(df[c].value_counts(dropna=False).sort_index().head(15))

## 4. Andre variabler vi trenger
Søk etter nøkkelord, og velg så de riktige ut fra kodeboken.

NHIS lagrer ofte `Refused`, `Not ascertained` og `Don't know` som tallkoder (f.eks. 7, 8, 9 eller 97, 98, 99). De må gjøres om til manglende verdier i rensingen, ellers blir gjennomsnittet feil.

In [ ]:
def find(*keywords):
    return [c for c in df.columns if any(k in c.upper() for k in keywords)]

print("Alder/kjønn:   ", find("AGEP", "SEX"))
print("Aktivitet:     ", find("PA", "EXER", "MODMIN", "VIGMIN")[:25])
print("Psykisk helse: ", find("ANX", "DEP", "PHQ", "GAD", "WORRY")[:25])
print("Inntekt/utd.:  ", find("INC", "POVRATIO", "EDUC", "EMPL"))
print("BMI/røyk/alk.: ", find("BMI", "SMK", "CIG", "ALC"))
print("Vekt:          ", find("WTFA"))

## 5. Manglende verdier (oversikt)

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
missing[missing > 0].head(20)

## Neste steg
1. Velg 15–25 variabler (`usecols`) ut fra kodeboken.
2. Rens: gjør kodene for «Refused / Don't know» om til manglende, og fjern urealistiske verdier.
3. Skriv ned hver beslutning (hva og hvorfor).
4. Lagre renset data i `data/processed/`.